#**Personal Protective Equipment Detection Project**

**Training YOLO11 Object Detection Model on a Custom Dataset for PPE Detection**

* **Dataset Details:**
  * **Project Name:** PPE_Detection Computer Vision Model
  * **Dataset Size:** Contains **3,290 images** with bounding box annotations for safety gear.

I explored several datasets for my PPE detection project and found this one to be the most suitable and reliable for my requirements. Therefore, I decided to use it for my project. I found this dataset on the Roboflow website, and I have shared the link below for your review.

**Dataset Link:** https://universe.roboflow.com/project-uyrxf/ppe_detection-v1x3l/dataset/2

### 🖥️ Hardware Acceleration & GPU Verification

Before executing heavy deep learning tasks, we run the system interface command to verify our hardware accelerator profile.

* **Purpose:** Confirms the allocated GPU hardware profile, checks temperature constraints, and monitors active Video RAM (VRAM) overhead.
* **Why it matters:** Ensures the environment is properly utilizing CUDA-enabled acceleration for accelerated YOLO training speeds.


In [ ]:
!nvidia-smi

### 📦 Environment Setup & Library Installation

We initialize our workspace by installing the official **Ultralytics** library.

* **Purpose:** Installs the core Python framework required to initialize, fine-tune, and deploy YOLO architecture variants.
* **Dependencies:** Automatically sets up deep learning backend wrappers (including PyTorch and CUDA bindings) to guarantee efficient execution on our GPU hardware.


In [ ]:
!pip install ultralytics

### 🔍 Environment Diagnostic & GPU Verification

We execute the native Ultralytics check utility to perform a complete system diagnostic before processing our image and video pipelines.

* **Purpose:** Validates the exact software framework version, cross-references internal dependencies, and confirms proper CUDA acceleration linkages.
* **Why it matters:** Guarantees that our deep learning model has uninterrupted access to the GPU infrastructure, preventing runtime slowdowns during inference execution.


In [ ]:
import ultralytics
ultralytics.checks()

### 🛠️ Module Architecture & Library Dependencies

We import the essential framework components needed to load our neural network architecture and visualize pipeline outputs.

* **YOLO (Ultralytics):** The primary object detection class used to construct, load, and execute predictions using our custom weights.
* **Image (IPython.display):** A specialized notebook rendering utility that lets us display visual training charts, metric evaluations, and prediction images inline.


In [ ]:
from ultralytics import YOLO
from IPython.display import Image

### 📥 Automated Dataset Ingestion via Roboflow API

The Roboflow Python SDK is used to securely authenticate and download the PPE Detection dataset directly into the runtime environment.

- **Authentication:** The user is prompted to enter their Roboflow API key using `getpass()`, preventing the key from being displayed during input.
- **Project Access:** The code connects to the `ppe_detection-v1x3l` project in the `project-uyrxf` workspace.
- **Version Selection:** Version 2 is selected to ensure consistency throughout the project.
- **Dataset Download:** The dataset is downloaded in YOLOv11 format, with images, annotations, and the `data.yaml` configuration organized into the required directory structure.

This automated process simplifies dataset preparation and enables the dataset to be used directly for training and evaluating the PPE detection model.

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
from getpass import getpass

# Ask the user to enter their Roboflow API key
api_key = getpass("Paste your Roboflow API key: ")

# Connect to Roboflow
rf = Roboflow(api_key=api_key)

# Load your PPE Detection project
project = rf.workspace("project-uyrxf").project("ppe_detection-v1x3l")

# Select dataset version 2
version = project.version(2)

# Download the dataset in YOLOv11 format
dataset = version.download("yolov11")

print("Dataset downloaded successfully!")
print("Dataset location:", dataset.location)

### 📂 Dataset Location Check

Prints the local folder path where Roboflow saved the downloaded dataset.

* **Purpose:** Confirms the download succeeded and shows the directory that holds the `train`, `valid` and `test` splits plus `data.yaml`.
* **Why it matters:** This path is reused as `{dataset.location}` in the training, validation and prediction commands below.

In [ ]:
dataset.location

### 🚀 Custom YOLO11n Neural Network Training Loop

We launch the model training loop to optimize our convolutional backbone weights on the custom personal protective equipment dataset.

* **Transfer Learning Foundation:** We load the lightweight `yolo11n.pt` pre-trained baseline framework to utilize robust feature extraction features.
* **Optimization Iterations:** The network parameters are exposed to the target training data subset for **40 complete epochs**, allowing the gradient descent optimizer to minimize loss metrics across localized bounding boxes.
* **Dimensional Constraints:** Input features are locked to a standard **640-pixel resolution scale** to strike a reliable balance between fine feature recognition (e.g., eye wear, gloves) and hardware memory limits.


In [ ]:
!yolo task=detect mode=train data={dataset.location}/data.yaml \
model="yolo11n.pt" \
epochs=40 \
imgsz=640


### 📈 Verification of Performance Curve Renderings

We execute a dedicated Python script to display our model's performance curves generated during the validation pass.

* **Recall-Confidence Curve:** Visualizes how effectively the network minimizes false negatives across classes as confidence criteria constraints are tightened.
* **Precision-Confidence Curve:** Demonstrates the purity of the model's classifications, indicating that as our operational threshold matches a higher confidence mark, the positive detection accuracy approaches a clean 100%.
* **Implementation Note:** Utilizing the `IPython.display` module avoids filesystem data-stream mismatches and allows direct visual inspection of data trends within the local workspace.


In [ ]:
from IPython.display import Image, display

# Display the Recall curve
print("--- Recall Curve ---")
display(Image("/content/runs/detect/train-2/BoxR_curve.png", width=600))

# Display the Precision curve
print("\n--- Precision Curve ---")
display(Image("/content/runs/detect/train-2/BoxP_curve.png", width=600))



The model shows strong overall performance.  
- Recall remains high (>0.8) for most classes until confidence exceeds 0.6, indicating good detection coverage.  
- Precision steadily improves with confidence, reaching near 1.0 around 0.9, showing reliable predictions at higher thresholds.  
- Protective Helmet and Eye Wear perform best, while Glove and Boots drop earlier, suggesting room for improvement in those classes.

##**Confusion Matrix**

### 🧮 Confusion Matrix (Raw Counts)

Displays the confusion matrix saved by YOLO after training.

* **Purpose:** Shows, for every PPE class, how many objects were detected correctly (diagonal) versus confused with another class or with background (off-diagonal).
* **Why it matters:** Reveals which classes the model mixes up and where it misses objects entirely.

In [ ]:
Image("/content/runs/detect/train-2/confusion_matrix.png", width=600)

### 🧮 Confusion Matrix (Normalized)

Displays the same confusion matrix, but scaled so each column sums to 1.

* **Purpose:** Converts raw counts into per-class proportions, making classes with very different sample sizes directly comparable.
* **Why it matters:** Easier to read class-wise accuracy and to spot weak classes in an imbalanced dataset.

In [ ]:
Image("/content/runs/detect/train-2/confusion_matrix_normalized.png", width=600)

The confusion matrices show that the model performs well overall but with some class imbalance.  
- High diagonal values in both matrices indicate strong accuracy for most protective equipment classes.  
- Normalized results show the model predicts Gloves and Protective Helmets most reliably (≈0.85 and 0.79).  
- Eye Wear and Dust Mask have lower normalized accuracy (≈0.48 and 0.69), suggesting these classes are more frequently misclassified.

### 🏷️ Dataset Label Distribution

Displays the `labels.jpg` summary generated by YOLO before training.

* **Purpose:** Shows the number of instances per class, bounding-box size/position overlays, and heatmaps of object centers and dimensions.
* **Why it matters:** Highlights class imbalance and spatial bias in the dataset, which directly affect model performance.

In [ ]:
Image("/content/runs/detect/train-2/labels.jpg", width=600)

This visualization summarizes the dataset distribution for object detection training.  
- The bar chart shows **Safety Vest**, **Protective Helmet**, and **Protective Boots** dominate the dataset, while **Shield** and **Eye Wear** have fewer samples, indicating class imbalance.  
- The bounding box overlay suggests varied object sizes and positions, confirming diverse spatial coverage.  
- The heatmaps reveal most objects cluster near the image center with moderate width–height ratios, typical for person-centered detection tasks.

### 📉 Training & Validation Metrics Overview

Displays `results.png`, the full training log plotted per epoch.

* **Purpose:** Plots box/class/DFL loss for training and validation, along with precision, recall, mAP50 and mAP50-95.
* **Why it matters:** Confirms the model is converging and helps detect overfitting or under-training.

In [ ]:
Image("/content/runs/detect/train-2/results.png", width=600)

These training results show the model is learning effectively.  
- All loss curves (box, class, object) steadily decrease for both training and validation, indicating consistent convergence.  
- Precision and mAP metrics rise sharply, stabilizing near high values, reflecting strong detection accuracy.  
- The upward trend in mAP50 and mAP50–95 confirms improved localization and classification performance across epochs.

###In this step, we are inspecting visual proof of our model's training and prediction data to make sure the AI is learning correctly.
Instead of looking at raw text or number percentages, we are checking the actual image batches that the YOLO model processed.

### 🖼️ Training Batch Sample

Displays `train_batch0.jpg`, a mosaic of images and ground-truth boxes fed to the model during training.

* **Purpose:** Verifies that annotations were loaded correctly and shows the augmentations (mosaic, scaling, flips) applied.

In [ ]:
Image("/content/runs/detect/train/train_batch0.jpg", width=600)

### 🔎 Validation Predictions – Batch 0

Displays the model's predictions on the first validation batch.

* **Purpose:** Visually compares predicted boxes and confidence scores against what is actually in the images.

In [ ]:
Image("/content/runs/detect/train-2/val_batch0_pred.jpg", width=600)

### 🔎 Validation Predictions – Batch 1

Displays predictions on the second validation batch for a wider qualitative check of detection quality.

In [ ]:
Image("/content/runs/detect/train-2/val_batch1_pred.jpg", width=600)

### 🔎 Validation Predictions – Batch 2

Displays predictions on the third validation batch, completing the visual review of how the model performs on unseen images.

In [ ]:
Image("/content/runs/detect/train-2/val_batch2_pred.jpg", width=600)

**Step # 06 Download the Model Weights from the Google Drive**

### ⬇️ Download Pre-Trained Weights (`best.pt`)

Uses `gdown` to download the model weights file from Google Drive.

* **Purpose:** Fetches `best.pt`, the fine-tuned YOLO weights used for validation and inference in the following steps.
* **`confirm=t`:** Bypasses Google Drive's large-file confirmation prompt.

In [ ]:
!gdown "https://drive.google.com/uc?id=1iuIB_3vKBunhyxEPhuecWPAWgVY71BNF&confirm=t"

The `best.pt` file downloaded above is a highly optimized "brain" obtained from a public repository.

* **Data Collection:** The creators compiled a dataset containing thousands of images specifically focused on **Personal Protective Equipment (PPE)**.
* **Rigorous Training:** They dedicated hours—potentially days—of computational power to train a **YOLO AI model** on their local systems.
* **Purpose:** Instead of training a model from scratch, we are leveraging these pre-trained, fine-tuned weights to achieve **more optimized, faster, and highly accurate detection** for our project.

**Step # 07  Validate Fine-Tuned Model**

### ✅ Model Validation

Runs YOLO in **validation mode** with `best.pt` on the dataset defined in `data.yaml`.

* **Purpose:** Computes the official evaluation metrics: Precision, Recall, mAP50 and mAP50-95, overall and per class.
* **Why it matters:** Gives quantitative proof of how accurate the fine-tuned model is.

In [ ]:
!yolo task=detect mode=val model="best.pt" data={dataset.location}/data.yaml

**Step # 08 Inference with Custom Model on Images**


### 🖼️ Run Inference on Test Images

Runs YOLO in **predict mode** on every image in the test folder.

* **`conf=0.25`:** Only detections with at least 25% confidence are kept.
* **`save=True`:** Saves annotated images (boxes + labels) to `runs/detect/predict*`.

In [ ]:
!yolo task=detect mode=predict model= best.pt conf=0.25 source={dataset.location}/test/images save=True

### 👀 Display Sample Prediction Results

Finds the most recent `predict` output folder and shows a few annotated test images.

* **How it works:** `glob` lists the result images, `os.path.getmtime` picks the newest folder, and `[1:4]` selects three images to display inline.
* **Purpose:** Quick visual inspection of detections without opening files manually.

In [ ]:
import glob
import os
from IPython.display import Image as IPyImage, display

latest_folder = max(glob.glob('/content/runs/detect/predict*/'), key=os.path.getmtime)
for img in glob.glob(f'{latest_folder}/*.jpg')[1:4]:
    display(IPyImage(filename=img, width=600))
    print("\n")

### ⬇️ Download a Custom Test Image

Downloads `image2.jpg` from Google Drive so the model can be tested on an image outside the dataset.

In [ ]:
!gdown "https://drive.google.com/uc?id=1ACZp9gmtjEdIY-SHp8K0xdR0D8Gjd3j3&confirm=t"

### 🖼️ Run Inference on the Custom Image

Runs the model on `image2.jpg` with a 0.25 confidence threshold and saves the annotated output.

In [ ]:
!yolo task=detect mode=predict model= best.pt conf=0.25 source=image2.jpg save=True

### 👀 View the Custom Image Result

Displays the annotated output image produced by the previous cell, showing detected PPE items with class labels and confidence scores.

In [ ]:
Image("/content/runs/detect/predict-2/image2.jpg", width=600)

**Step # 08 Inference with Custom Model on Videos**

### ⬇️ Download Test Video 1

Downloads the first sample workplace video (`PPE_Part1.mp4`) from Google Drive.

In [ ]:
!gdown "https://drive.google.com/uc?id=1Nyl18zr7zOJHh8uHtSic3dsJSY4LI_t-&confirm=t"

### ⬇️ Download Test Video 2

Downloads the second sample workplace video (`PPE_Part2.mp4`) from Google Drive.

In [ ]:
!gdown "https://drive.google.com/uc?id=1SxcrDanPAr-PZfJjWBxoa2t8lUSM2dZn&confirm=t"

### 🎥 Run Inference on Video 1

Runs the model frame-by-frame on `PPE_Part1.mp4` and saves the annotated video.

* **Note:** YOLO saves the output as an `.avi` file inside `runs/detect/predict*`.

In [ ]:
!yolo task=detect mode=predict model= best.pt conf=0.25 source="PPE_Part1.mp4" save=True

### 🧹 Remove Old Compressed Video

Deletes any previously created `result_compressed.mp4` so the next step writes a fresh file and never shows an outdated result.

In [ ]:
!rm '/content/result_compressed.mp4'

### 📺 Compress & Play Video 1 Result

Makes the annotated video playable inside the notebook.

* **Locate:** Automatically finds the newest `predict` folder and the `PPE_Part1.avi` output.
* **Compress:** Uses `ffmpeg` to re-encode the `.avi` into browser-friendly H.264 `.mp4`.
* **Display:** Encodes the video as base64 and embeds it in an HTML `<video>` player. A clear error message is printed if the file is missing.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

# 1. Automatically search for the processed video file
possible_dirs = sorted([d for d in os.listdir('/content/runs/detect') if d.startswith('predict')], reverse=True)

if not possible_dirs:
    raise FileNotFoundError("Could not find any 'predict' folders in /content/runs/detect. Did you run the prediction cell?")

# Pick the most recent predict folder (e.g., predict3, predict2, or predict)
latest_predict_dir = os.path.join('/content/runs/detect', possible_dirs[0])
save_path = os.path.join(latest_predict_dir, 'PPE_Part1.avi')

print(f"Looking for processed video at: {save_path}")

# 2. Path where the compressed video will be saved
compressed_path = "/content/result_compressed.mp4"

# Remove any old file if it exists
if os.path.exists(compressed_path):
    os.remove(compressed_path)

# 3. Correctly format and run the ffmpeg command using f-string
print("Compressing video... please wait...")
os.system(f"ffmpeg -y -i {save_path} -vcodec libx264 {compressed_path}")

# 4. Display the video inside Colab
if os.path.exists(compressed_path):
    mp4 = open(compressed_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=500 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("Error: Compression failed. Please check if your video file actually exists in the predict folder.")


### 🎥 Run Inference on Video 2

Runs the model on `PPE_Part2.mp4` with a 0.25 confidence threshold and saves the annotated video.

In [ ]:
!yolo task=detect mode=predict model=best.pt conf=0.25 source="PPE_Part2.mp4" save=True

### 🧹 Remove Old Compressed Video

Clears the previous compressed file so the Video 2 result is written cleanly.

In [ ]:
!rm '/content/result_compressed.mp4'

### 📺 Compress & Play Video 2 Result

Same process as Video 1: converts the annotated `.avi` to H.264 `.mp4` with `ffmpeg` and shows it in an inline video player.

* **Note:** The output path (`predict-4`) is hard-coded, so adjust it if your prediction folder has a different name.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import os

# Input video path
save_path = '/content/runs/detect/predict-4/PPE_Part2.avi'

# Compressed video path
compressed_path = "/content/result_compressed.mp4"

# Run the compression command (added -y to overwrite cleanly)
print("Compressing video... please wait...")
os.system(f"ffmpeg -y -i {save_path} -vcodec libx264 {compressed_path}")

# Show video cleanly using display()
if os.path.exists(compressed_path):
    mp4 = open(compressed_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    # Using display() explicitly makes the player appear!
    display(HTML(f"""
    <video width="600" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
    print("Video rendered successfully!")
else:
    print("Error: Video compression failed. Check if the file exists in /content/runs/detect/predict-4/")


### ⬆️ Test on Your Own Video (Upload → Detect → Play)

An end-to-end pipeline for any video from your computer:

1. **Upload:** `files.upload()` opens a file picker and stores the chosen MP4.
2. **Detect:** Runs the YOLO model (`best.pt`, `conf=0.25`) on the uploaded video.
3. **Locate:** Finds the newest `predict` folder and the matching `.avi` output.
4. **Compress:** Converts the result to H.264 `.mp4` with `ffmpeg`.
5. **Display:** Shows the final video in an inline player, or prints an error if something failed.

In [ ]:
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode
import os

# Step 1: Upload the MP4 file from your PC
print("Please select your MP4 video file from your computer:")
uploaded = files.upload()

# Get the name of the file you uploaded
uploaded_file_name = list(uploaded.keys())[0]
print(f"Successfully uploaded: {uploaded_file_name}")

# Step 2: Run the YOLO Object Detection model on your video
print("\nRunning safety gear detection model... please wait...")
# We save the output text log to find the exact folder name
os.system(f"yolo task=detect mode=predict model=best.pt conf=0.25 source='{uploaded_file_name}' save=True")

# Step 3: Find the newest prediction folder automatically
possible_dirs = sorted([d for d in os.listdir('/content/runs/detect') if d.startswith('predict')], reverse=True)
latest_predict_dir = os.path.join('/content/runs/detect', possible_dirs[0])

# YOLO changes the extension from .mp4 to .avi when saving
video_base_name = os.path.splitext(uploaded_file_name)[0]
processed_avi_path = os.path.join(latest_predict_dir, f"{video_base_name}.avi")

# Step 4: Compress the output video so it can play in your browser
compressed_path = "/content/my_pc_video_detected.mp4"
print("\nCompressing the results video...")
os.system(f"ffmpeg -y -i '{processed_avi_path}' -vcodec libx264 '{compressed_path}'")

# Step 5: Display the final video player
if os.path.exists(compressed_path):
    print("\n--- Displaying Your Checked Video ---")
    mp4_bytes = open(compressed_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    display(HTML(f"""
    <video width="600" controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
else:
    print("\nError: Could not display video. Check if the model ran successfully.")


##**Web Interface Development**

###Step 1: Install Gradio

### 📦 Install Gradio

Installs **Gradio** (web UI framework) and **Ultralytics** quietly (`-q`).

* **Purpose:** Gradio turns the Python detection functions into an interactive web app with no front-end coding.

In [ ]:
!pip install -q gradio ultralytics

#### Step 2: Creating PPE detection web application

### 🌐 Build & Launch the PPE Detection Web App

This cell builds the full Gradio application in six parts:

1. **Load model:** Searches `/content` for the newest `best.pt`, loads it with YOLO, and prints the class names.
2. **`detect_image`:** Runs detection on one image at the chosen confidence, draws the boxes, and converts BGR→RGB for display.
3. **`detect_video`:** Reads the video frame by frame with OpenCV, runs detection on each frame, writes the annotated frames to a new video, then converts it to H.264 with `ffmpeg` so browsers can play it.
4. **`detect_live`:** Same detection logic applied to webcam frames for real-time monitoring.
5. **Interface:** A Gradio `Blocks` layout with three tabs (Image, Video, Live Webcam), each with input/output components and a confidence slider (0.1–0.9, default 0.4).
6. **Launch:** `demo.queue().launch(share=True)` starts the app and generates a public shareable link.

In [ ]:
import gradio as gr
import cv2
import os
import glob
import tempfile
import numpy as np
from ultralytics import YOLO

# ==================================
# 1. LOAD TRAINED MODEL
# ==================================

model_files = glob.glob(
    "/content/**/best.pt",
    recursive=True
)

if not model_files:
    raise FileNotFoundError(
        "Trained best.pt model not found!"
    )

model_path = max(
    model_files,
    key=os.path.getmtime
)

model = YOLO(model_path)

print("Model loaded:", model_path)
print("Classes:", model.names)


# ==================================
# 2. IMAGE DETECTION
# ==================================

def detect_image(image, confidence):

    if image is None:
        return None

    results = model.predict(
        image,
        conf=confidence,
        imgsz=640,
        verbose=False
    )

    annotated = results[0].plot()

    return cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )


# ==================================
# 3. VIDEO DETECTION
# ==================================

def detect_video(video_path, confidence):

    if video_path is None:
        return None

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise gr.Error("Unable to open video.")

    fps = cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        fps = 25

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = tempfile.mktemp(
        suffix=".mp4"
    )

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    out = cv2.VideoWriter(
        output_path,
        fourcc,
        fps,
        (width, height)
    )

    if not out.isOpened():
        cap.release()
        raise gr.Error("Unable to create output video.")

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model.predict(
            frame,
            conf=confidence,
            imgsz=640,
            verbose=False
        )

        annotated = results[0].plot()

        out.write(annotated)

    cap.release()
    out.release()

    # Convert to browser-compatible H.264
    compatible_path = tempfile.mktemp(
        suffix=".mp4"
    )

    import subprocess

    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", output_path,
            "-c:v", "libx264",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            compatible_path
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE
    )

    return compatible_path


# ==================================
# 4. LIVE WEBCAM DETECTION
# ==================================

def detect_live(frame, confidence):

    if frame is None:
        return None

    results = model.predict(
        frame,
        conf=confidence,
        imgsz=640,
        verbose=False
    )

    annotated = results[0].plot()

    return cv2.cvtColor(
        annotated,
        cv2.COLOR_BGR2RGB
    )


# ==================================
# 5. CREATE WEB INTERFACE
# ==================================

with gr.Blocks(
    title="AI PPE Detection System"
) as demo:

    gr.Markdown(
        """
        # AI-Powered PPE Detection System

        ### Real-Time Workplace Safety Monitoring

        **Developed using YOLO and Computer Vision**

        Detect Personal Protective Equipment
        using images, recorded videos,
        or a live webcam.
        """
    )

    with gr.Tabs():

        # IMAGE TAB
        with gr.Tab("Image Detection"):

            with gr.Row():

                image_input = gr.Image(
                    label="Upload Image",
                    type="numpy"
                )

                image_output = gr.Image(
                    label="Detection Results"
                )

            image_conf = gr.Slider(
                0.1, 0.9,
                value=0.4,
                label="Confidence"
            )

            image_button = gr.Button(
                "Detect PPE",
                variant="primary"
            )

            image_button.click(
                detect_image,
                inputs=[
                    image_input,
                    image_conf
                ],
                outputs=image_output
            )

        # VIDEO TAB
        with gr.Tab("Video Detection"):

            video_input = gr.Video(
                label="Upload Workplace Video"
            )

            video_conf = gr.Slider(
                0.1, 0.9,
                value=0.4,
                label="Confidence"
            )

            video_button = gr.Button(
                "Process Video",
                variant="primary"
            )

            video_output = gr.Video(
                label="Processed PPE Detection Video"
            )

            video_button.click(
                detect_video,
                inputs=[
                    video_input,
                    video_conf
                ],
                outputs=video_output
            )

        # LIVE WEBCAM TAB
        with gr.Tab("Live Webcam Detection"):

            gr.Markdown(
                """
                ### Live PPE Monitoring

                Allow camera access to start
                real-time PPE detection.
                """
            )

            webcam_input = gr.Image(
                sources=["webcam"],
                type="numpy",
                label="Live Camera",
                streaming=True
            )

            webcam_conf = gr.Slider(
                0.1, 0.9,
                value=0.4,
                label="Confidence"
            )

            webcam_output = gr.Image(
                label="Live PPE Detection"
            )

            webcam_input.stream(
                fn=detect_live,
                inputs=[
                    webcam_input,
                    webcam_conf
                ],
                outputs=webcam_output,
                stream_every=0.2,
                time_limit=120
            )

    gr.Markdown(
        """
        ---
        **Final Year AI Project — PPE Detection System**

        Powered by Ultralytics YOLO
        """
    )


# ==================================
# 6. LAUNCH WEBSITE
# ==================================

demo.queue().launch(
    share=True,
    debug=True
)

# Results, Improvements and Conclusion

## 1. Results

In this project, I developed a PPE Detection system using the YOLO11n model. The main purpose of this project was to detect personal protective equipment in images and videos using computer vision.

First, I downloaded the dataset from Roboflow and used it to train the model. After training, I checked the model performance using the training graphs, confusion matrix, and validation results. I also tested the model on images and videos to see how well it could detect PPE items.

I also created a simple web interface using Gradio, which allows users to upload images and check the detection results.

From this project, I learned how to prepare a dataset, train an object detection model, evaluate its performance, and use the trained model for predictions. However, there are still some areas where I can improve the system.

## 2. Improvements That Can Be Made

### 1. Improve the Dataset

One improvement I can make is to increase the number of images for classes that have fewer samples. This may help the model learn better and improve its detection performance for those classes.

### 2. Improve Detection Accuracy

I can experiment with different YOLO11 models, such as YOLO11s, and compare their results with YOLO11n. This will help me understand whether a larger model can provide better accuracy for my project.

### 3. Adjust the Confidence Threshold

Currently, I use a confidence threshold of 0.25 for detection. I can test different threshold values and compare the results to find a suitable value that reduces incorrect detections without missing too many actual PPE items.

### 4. Improve Performance in Different Conditions

The model may not perform equally well in every situation. For example, poor lighting, small objects, and partially hidden PPE items can make detection difficult. I can improve this by adding more diverse images and applying suitable data augmentation techniques.

### 5. Improve Video Detection

I can further improve the video detection part by testing the model on different videos and measuring how quickly it processes each frame. This will help me understand whether the model can be used for real-time detection.

### 6. Improve the Web Interface

I can make the Gradio interface more user-friendly by adding options to adjust the confidence threshold, display detection confidence scores, and download the output images or videos.

## 3. Future Work

In the future, I would like to improve the model by training it on a larger and more balanced dataset. I can also compare different model sizes and evaluate their accuracy and speed.

Another improvement could be to detect workers who are missing required safety equipment. For example, the system could identify situations where a worker is not wearing a helmet or safety vest. However, this would require additional logic to associate detected equipment with individual workers.

I can also explore deploying the model on a camera or an edge device to test its performance in a real working environment.

## 4. Conclusion

Overall, this project helped me understand how computer vision can be used for PPE detection. I learned how to download and prepare a dataset, train a YOLO11n model, evaluate its performance, and use it for image and video detection.

I also developed a Gradio interface to make the system easier to use. Although the current system provides a starting point for PPE detection, there is still room for improvement in detection accuracy, performance, and usability.

By working on the improvements discussed above, I can make the system more reliable and useful for workplace safety monitoring.